# Vibration sensors — per-sensor 5-minute cadence and smoothed RMS

Load the combined CSV, split by `SENSOR_DESC`, sort each group by `TIMESTAMP`, keep rows whose time since the **previous** row is about **5 minutes** (drop gaps and irregular spacing), then plot **Acceleration RMS** with a **200-point centered rolling mean**.

Charts are **Plotly**: pan and zoom on the axes, box-zoom from the mode bar, and the **range slider** under the x-axis for quick scrubbing.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# Interactive figures: pan/zoom in the plot, or use the range slider under the x-axis.
pio.renderers.default = "browser"
# In Jupyter/VS Code, if you prefer inline widgets: try "notebook_connected" or "vscode".

REPO_ROOT = Path(r"C:\Users\NGYX\Desktop\Murata_NGYX")
DATA_PATH = REPO_ROOT / "data" / "Vibration sensors _ 2022 to 2026.csv"

SENSOR_COL = "SENSOR_DESC"
TIME_COL = "TIMESTAMP"
VALUE_COL = "Acceleration RMS"

STEP_SECONDS = 300.0
STEP_TOL_SECONDS = 30.0
ROLL_WINDOW = 200
# Break smoothed line when consecutive kept rows are farther apart than nominal 5 min step.
LINE_BREAK_IF_DT_GT = pd.Timedelta(seconds=STEP_SECONDS + 2 * STEP_TOL_SECONDS)

In [4]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Missing data file: {DATA_PATH}")

df_all = pd.read_csv(DATA_PATH)
for c in (SENSOR_COL, TIME_COL, VALUE_COL):
    if c not in df_all.columns:
        raise ValueError(f"Expected column {c!r} in CSV. Got: {list(df_all.columns)}")

print(f"Loaded {len(df_all):,} rows, {df_all[SENSOR_COL].nunique()} distinct {SENSOR_COL!r}")

Loaded 814,179 rows, 9 distinct 'SENSOR_DESC'


C:\Users\NGYX\AppData\Local\Temp\ipykernel_13360\2056067563.py:4: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all = pd.read_csv(DATA_PATH)


In [5]:
def filter_incoming_five_minutes(
    frame: pd.DataFrame,
    ts_col: str,
    nominal_s: float,
    tol_s: float,
) -> pd.DataFrame:
    """Keep row i iff TIMESTAMP[i] - TIMESTAMP[i-1] ≈ nominal_s ± tol_s (row 0 iff first step matches)."""
    ts = frame[ts_col]
    dt_sec = ts.diff().dt.total_seconds().to_numpy()
    step_ok = np.zeros(len(ts), dtype=bool)
    step_ok[1:] = np.abs(dt_sec[1:] - float(nominal_s)) <= float(tol_s)
    keep = np.zeros(len(ts), dtype=bool)
    if len(keep) > 0:
        keep[0] = len(ts) > 1 and bool(step_ok[1])
        if len(ts) > 1:
            keep[1:] = step_ok[1:]
    out = frame.loc[keep].reset_index(drop=True)
    return out


def prepare_sensor_frame(raw_sensor: pd.DataFrame) -> pd.DataFrame:
    g = raw_sensor.copy()
    g[TIME_COL] = pd.to_datetime(g[TIME_COL], dayfirst=True, format="mixed")
    g = g.sort_values(TIME_COL).reset_index(drop=True)
    n_before = len(g)
    g = filter_incoming_five_minutes(g, TIME_COL, STEP_SECONDS, STEP_TOL_SECONDS)
    g.attrs["dropped_non_uniform"] = n_before - len(g)
    y = pd.to_numeric(g[VALUE_COL], errors="coerce")
    smooth = y.rolling(window=ROLL_WINDOW, center=True, min_periods=1).mean()
    gap = g[TIME_COL].diff() > LINE_BREAK_IF_DT_GT
    smooth_plot = smooth.mask(gap.fillna(False), np.nan)
    g = g.assign(
        _y_raw=y,
        _y_smooth=smooth,
        _y_smooth_plot=smooth_plot,
    )
    return g


sensor_tables: dict[str, pd.DataFrame] = {}
for sensor_name, chunk in df_all.groupby(SENSOR_COL, sort=False):
    sensor_tables[str(sensor_name)] = prepare_sensor_frame(chunk)

for name, g in sensor_tables.items():
    dropped = int(g.attrs.get("dropped_non_uniform", 0))
    print(
        f"{name[:72]}{'…' if len(name) > 72 else ''}: "
        f"kept {len(g):,} rows (dropped {dropped:,} non–5-min-from-previous)"
    )

AHU 2-9 Blower DE Vibration X: kept 133,487 rows (dropped 15,547 non–5-min-from-previous)
AHU 4-4 Blower DE Vibration X: kept 145,440 rows (dropped 4,265 non–5-min-from-previous)
AHU 2-9 Blower DE V: kept 81,522 rows (dropped 2,810 non–5-min-from-previous)
AHU 2-9 motor NDE H: kept 63,992 rows (dropped 9,410 non–5-min-from-previous)
AHU 2-9 Blower DE A: kept 58,517 rows (dropped 3,278 non–5-min-from-previous)
AHU 2-9 Blower NDE V: kept 78,211 rows (dropped 4,706 non–5-min-from-previous)
AHU 2-9 Blower NDE A: kept 86,247 rows (dropped 5,281 non–5-min-from-previous)
AHU 2-9 Blower NDE H: kept 41,925 rows (dropped 14,682 non–5-min-from-previous)
AHU 2-9 motor DE H: kept 58,824 rows (dropped 6,035 non–5-min-from-previous)


In [6]:
for sensor_name, g in sensor_tables.items():
    short = sensor_name if len(sensor_name) <= 100 else sensor_name[:97] + "…"
    dropped = int(g.attrs.get("dropped_non_uniform", 0))
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=g[TIME_COL],
            y=g["_y_raw"],
            mode="markers",
            name=f"{VALUE_COL} (5 min from previous row)",
            marker=dict(size=4, opacity=0.35),
            hovertemplate="%{x}<br>%{y:.4f}<extra></extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=g[TIME_COL],
            y=g["_y_smooth_plot"],
            mode="lines",
            name=f"Rolling mean {ROLL_WINDOW} (centered)",
            line=dict(width=1.8, color="#EF553B"),
            connectgaps=False,
            hovertemplate="%{x}<br>smoothed=%{y:.4f}<extra></extra>",
        )
    )
    fig.update_layout(
        title=(
            f"{short}<br><sup>Δt from previous ≈ {STEP_SECONDS:.0f}s ± {STEP_TOL_SECONDS:.0f}s; "
            f"dropped {dropped:,} rows — drag to pan (or use toolbar), scroll wheel to zoom, range slider below</sup>"
        ),
        xaxis_title=TIME_COL,
        yaxis_title=VALUE_COL,
        template="plotly_white",
        hovermode="x unified",
        height=520,
        width=1200,
        dragmode="pan",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )
    fig.update_xaxes(rangeslider_visible=True)
    fig.show()